# Entrenar ilustraciones — SD 1.5 + LoRA (Colab Pro+)

Entrena un LoRA de **estilo plano** sobre Stable Diffusion 1.5, condicionado por **temática**, a partir
del CSV que produjo el indexador (`dataset_ilustraciones.csv`, 19,753 imágenes en 12 temáticas).

Diseño del pipeline:
- **Entrenamiento:** se condiciona solo por temática → el LoRA aprende el *estilo* del equipo y la temática
  es el **puente** con el modelo de cuentos (se entrena igual que como se infiere).
- **Generación:** temática (estilo + puente) **+ una escena corta del cuento** (título o frase) para que la
  ilustración sea *relevante* a esa historia. La escena la entiende el SD base; el estilo lo pone el LoRA.

Optimizado para A100 (Pro+): lote mayor + bf16. Checkpoints a Drive con reanudación automática.


## Paso 0 — Setup (GPU, dependencias, Drive)

In [ ]:
import os, sys
EN_COLAB = "google.colab" in sys.modules
print("En Colab:", EN_COLAB)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Sin GPU: Runtime > GPU (A100)"

if EN_COLAB:
    !pip install -q diffusers peft torchvision transformers accelerate safetensors pillow matplotlib
    # Colab trae un torchao viejo (0.10) que rompe a peft (exige >0.16). No lo usamos -> lo quitamos.
    !pip uninstall -q -y torchao
    from google.colab import drive
    drive.mount("/content/drive")


## Paso 1 — Configuración

`RUTA_DATASET_CSV` y `CARPETA_RAIZ` deben coincidir con lo que usó el indexador (¡ojo con el espacio en
`/MyDrive/ IMAGENES_CUENTOS`!).


In [ ]:
import os, random
import numpy as np
import torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # evita "image file is truncated" a mitad del entrenamiento

# === Entradas (deben coincidir con el indexador) ===
RUTA_DATASET_CSV = "/content/drive/MyDrive/ilustraciones_dataset/dataset_ilustraciones.csv"
CARPETA_RAIZ     = "/content/drive/MyDrive/IMAGENES_CUENTOS"   # <-- mismo string EXACTO que el indexador

LADO = 512   # Stable Diffusion trabaja a 512x512

# === Modelo / LoRA ===
MODELO_BASE  = "stable-diffusion-v1-5/stable-diffusion-v1-5"  # publico, sin token
RANGO_LORA   = 8         # un poco mas de capacidad que 4 (tienes ~20k imagenes)
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

# === Entrenamiento (afinado para A100) ===
LOTE               = 4       # A100 aguanta lote 4-8 a 512px. Si OOM, baja a 2.
USAR_BF16          = True    # acelera mucho en A100
PASOS              = 3000    # con LOTE=4 ve ~12k imagenes. Sube a 5000-6000 si quieres mas
LR                 = 1e-4
WARMUP             = 100
LOG_CADA           = 25
GUARDAR_CADA_PASOS = 250     # checkpoints a Drive (crash-safe)
LIMITE_CKPTS       = 3
SEED               = 42

# === Salida (Drive) — ruta propia para no pisar el modelo de cuentos ===
RUTA_SALIDA = "/content/drive/MyDrive/ilustraciones_modelo/sd15_estilo_lora"
RUTA_FINAL  = os.path.join(RUTA_SALIDA, "final")
os.makedirs(RUTA_SALIDA, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Dataset CSV   :", RUTA_DATASET_CSV, "| existe:", os.path.isfile(RUTA_DATASET_CSV))
print("Imagenes raiz :", CARPETA_RAIZ,     "| existe:", os.path.isdir(CARPETA_RAIZ))
print("Salida        :", RUTA_SALIDA)


## Paso 2 — Cargar dataset desde el CSV

In [ ]:
import csv
from collections import Counter

imagenes_csv = []
with open(RUTA_DATASET_CSV, encoding="utf-8-sig", newline="") as f:
    for fila in csv.DictReader(f):
        ruta = os.path.join(CARPETA_RAIZ, fila["recolector"], fila["archivo"])
        imagenes_csv.append({"ruta": ruta, "tematica": fila["tematica"],
                             "recolector": fila["recolector"]})

print(f"Filas en el CSV: {len(imagenes_csv)}")
print("\nImagenes por tematica:")
for tem, n in sorted(Counter(r["tematica"] for r in imagenes_csv).items(), key=lambda x: -x[1]):
    print(f"  {tem:<28} {n:>6}")


## Paso 3 — Verificar que los archivos existen

In [ ]:
imagenes_validas, no_encontradas = [], []
for r in imagenes_csv:
    (imagenes_validas if os.path.isfile(r["ruta"]) else no_encontradas).append(r)

print(f"Encontradas en disco : {len(imagenes_validas)}")
if no_encontradas:
    print(f"NO encontradas       : {len(no_encontradas)}")
    print("  (primeras 5):", [r['ruta'] for r in no_encontradas[:5]])
    print("  -> Revisa CARPETA_RAIZ o que las subcarpetas no se hayan renombrado.")
else:
    print("Todas las imagenes del CSV estan en disco. Listo para entrenar.")


## Paso 4 — Resumen por temática

In [ ]:
conteo = Counter(img["tematica"] for img in imagenes_validas)
print("Imagenes por tematica:\n")
for tem, n in sorted(conteo.items()):
    print(f"  {tem:<30} {n:>6}")
print(f"\n  {'TOTAL':<30} {sum(conteo.values()):>6}  | tematicas: {len(conteo)}")


## Paso 5 — Preparar dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ImagenesEstiloDataset(Dataset):
    def __init__(self, imagenes_info):
        self.imagenes = imagenes_info
        self.transform = transforms.Compose([
            transforms.Resize((LADO, LADO)),
            transforms.CenterCrop(LADO),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self): return len(self.imagenes)
    def __getitem__(self, i):
        info = self.imagenes[i]
        try:
            img = Image.open(info["ruta"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (LADO, LADO), (255, 255, 255))  # robustez ante imagen rota
        tensor = self.transform(img)
        # ENTRENAMIENTO: solo tematica (ancla de estilo + puente con cuentos)
        prompt = f"ilustracion plana a color sobre {info['tematica'].replace('_', ' ')}"
        return {"pixel_values": tensor, "prompt": prompt}

dataset = ImagenesEstiloDataset(imagenes_validas)
print(f"Dataset listo: {len(dataset)} imagenes a {LADO}x{LADO}")
print(f"Prompt ejemplo: '{dataset[0]['prompt']}'")


## Paso 6 — Cargar Stable Diffusion y aplicar LoRA

In [ ]:
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model

print(f"Cargando {MODELO_BASE}...")
pipe = StableDiffusionPipeline.from_pretrained(MODELO_BASE, torch_dtype=torch.float32)
unet = pipe.unet.to("cuda")
vae = pipe.vae.to("cuda")
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to("cuda")
noise_scheduler = DDPMScheduler.from_pretrained(MODELO_BASE, subfolder="scheduler")

lora_config = LoraConfig(r=RANGO_LORA, lora_alpha=LORA_ALPHA,
                         target_modules=["to_q", "to_v", "to_k", "to_out.0"],
                         lora_dropout=LORA_DROPOUT)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
print("Modelo listo para entrenar.")


## Paso 7 — Entrenar (lote A100 + bf16 + checkpoints/reanudación)

Guarda a Drive cada `GUARDAR_CADA_PASOS`. Si la sesión se cae, vuelve a correr desde el Paso 0 y retoma
el último checkpoint solo. En Pro+ puedes cerrar la pestaña (ejecución en segundo plano).


In [ ]:
import glob, shutil
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

usar_bf16 = USAR_BF16 and torch.cuda.is_bf16_supported()
print("Precision:", "bf16" if usar_bf16 else "fp32", "| lote:", LOTE)

def ultimo_checkpoint(carpeta):
    cks = [c for c in glob.glob(os.path.join(carpeta, "checkpoint-*")) if c.split("-")[-1].isdigit()]
    return max(cks, key=lambda p: int(p.split("-")[-1])) if cks else None

def guardar_checkpoint(carpeta, unet, optimizador, paso, perdidas, limite):
    ckpt = os.path.join(carpeta, f"checkpoint-{paso}")
    os.makedirs(ckpt, exist_ok=True)
    unet.save_pretrained(ckpt)
    torch.save({"optimizador": optimizador.state_dict(), "paso": paso, "perdidas": perdidas},
               os.path.join(ckpt, "estado.pt"))
    todos = sorted(glob.glob(os.path.join(carpeta, "checkpoint-*")), key=lambda p: int(p.split("-")[-1]))
    for viejo in todos[:-limite]:
        shutil.rmtree(viejo, ignore_errors=True)

cargador = DataLoader(dataset, batch_size=LOTE, shuffle=True, num_workers=2, drop_last=True)
optimizador = torch.optim.AdamW(unet.parameters(), lr=LR)

def factor_lr(p):
    if p < WARMUP: return p / max(1, WARMUP)
    prog = (p - WARMUP) / max(1, PASOS - WARMUP)
    return 0.5 * (1 + np.cos(np.pi * min(1.0, prog)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)

# --- Reanudacion ---
paso, perdidas = 0, []
ckpt = ultimo_checkpoint(RUTA_SALIDA)
if ckpt:
    print("Reanudando desde:", ckpt)
    set_peft_model_state_dict(unet, load_file(os.path.join(ckpt, "adapter_model.safetensors")))
    estado = torch.load(os.path.join(ckpt, "estado.pt"))
    optimizador.load_state_dict(estado["optimizador"]); paso = estado["paso"]; perdidas = estado["perdidas"]
    for _ in range(paso): scheduler.step()
else:
    print("Empezando desde cero.")

unet.train()
print(f"Entrenando hasta {PASOS} pasos (desde el paso {paso})...\n")
while paso < PASOS:
    for lote in cargador:
        if paso >= PASOS: break
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=usar_bf16):
            with torch.no_grad():
                latentes = vae.encode(lote["pixel_values"].to("cuda")).latent_dist.sample() * 0.18215
                tokens = tokenizer(list(lote["prompt"]), padding="max_length",
                                   max_length=tokenizer.model_max_length, truncation=True,
                                   return_tensors="pt").input_ids.to("cuda")
                encoder_hidden = text_encoder(tokens)[0]
            ruido = torch.randn_like(latentes)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                      (latentes.shape[0],), device="cuda").long()
            latentes_ruidosos = noise_scheduler.add_noise(latentes, ruido, timesteps)
            prediccion = unet(latentes_ruidosos, timesteps, encoder_hidden).sample
            perdida = torch.nn.functional.mse_loss(prediccion.float(), ruido.float())

        optimizador.zero_grad(); perdida.backward(); optimizador.step(); scheduler.step()
        paso += 1; perdidas.append(perdida.item())

        if paso % LOG_CADA == 0 or paso == 1:
            print(f"  Paso {paso}/{PASOS}  perdida={perdida.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
        if paso % GUARDAR_CADA_PASOS == 0:
            guardar_checkpoint(RUTA_SALIDA, unet, optimizador, paso, perdidas, LIMITE_CKPTS)
            print(f"  checkpoint guardado en el paso {paso}")

print("\nEntrenamiento terminado.")


## Paso 8 — Gráfica de pérdida

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(perdidas)
plt.title("Perdida durante el entrenamiento LoRA"); plt.xlabel("Paso"); plt.ylabel("Perdida (MSE)")
plt.tight_layout(); plt.show()


## Paso 9 — Guardar pesos LoRA finales

In [ ]:
os.makedirs(RUTA_FINAL, exist_ok=True)
unet.save_pretrained(RUTA_FINAL)
with open(os.path.join(RUTA_FINAL, "config_base.txt"), "w") as f:
    f.write(MODELO_BASE)
print("Pesos LoRA finales guardados en:", RUTA_FINAL)


## Paso 10 — Generar: temática (estilo) + escena del cuento (relevancia)

`generar(tema)` → ilustración genérica de la temática (estilo del equipo).
`generar(tema, escena="...")` → misma temática y estilo, pero relevante a ESE cuento. La `escena` sale del
título o de una frase corta del cuento (sujeto + lugar + acción, simple).


In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel
import matplotlib.pyplot as plt

pipe_test = StableDiffusionPipeline.from_pretrained(MODELO_BASE, torch_dtype=torch.float16, safety_checker=None)
pipe_test.unet = PeftModel.from_pretrained(pipe_test.unet, RUTA_FINAL)
pipe_test = pipe_test.to("cuda")

NEG = "fotografia, realista, 3d, texto, marca de agua, deforme"

def generar(tematica, escena=None, pasos=30, guidance=7.5):
    tema = tematica.replace("_", " ")
    prompt = f"ilustracion plana infantil a color sobre {tema}"
    if escena:
        prompt += f", {escena}, estilo caricatura infantil"
    return pipe_test(prompt=prompt, negative_prompt=NEG,
                     num_inference_steps=pasos, guidance_scale=guidance).images[0]

# Comparacion: solo tematica vs tematica + escena del cuento
ejemplos = [
    ("princesas_y_castillos", None),
    ("princesas_y_castillos", "princesa dormida"),
    ("espacio", None),
    ("espacio", "un pequeno robot mirando un planeta con anillos"),
]
fig, axes = plt.subplots(1, len(ejemplos), figsize=(20, 5))
for ax, (tema, escena) in zip(axes, ejemplos):
    ax.imshow(generar(tema, escena))
    ax.set_title(tema + ("\n+ escena" if escena else "\n(solo tema)"), fontsize=9)
    ax.axis("off")
plt.suptitle("Estilo por tematica (LoRA) + relevancia por escena del cuento")
plt.tight_layout(); plt.show()
